# 03 — re-score rung 47 on the FULL 6,252

Rung 47's three checkpoints, re-answered on the **whole `frame_ood_v1` validation set — 38
videos / 6,252 questions** instead of rung 42's 8 / 1,283. No training, no new data, inference
only.

**Why.** The 8-video instrument cannot resolve this effect. Run through
`frame.metrics.paired_delta_ci` on 2026-08-19, the epoch effect returned **no cell excluding
zero** (`ALL_ID` +0.0290 [−0.0145, +0.0705]). Interval half-width shrinks roughly with the
square root of the cluster count: √(38/8) ≈ 2.2×. Same effect, readable instrument.

**Why it is legal — measured, not argued.** Rung 47 trained on A2's corpus. Parsing
`rung47/corpus/train_mntpaths.jsonl` gives **exactly the 92 `train` videos: 0 of the 28
`val_id`, 0 of the 10 `val_ood`, 0 unrecognised.** Cell 3 re-runs that as a gate.

🔴 **Rung 42 may NOT be re-scored this way** — it trained on 30 of the 38. No control from it
appears here, and the numbers this notebook produces are **not comparable** to its 0.6744, nor
to rung 47's own published 0.6468 (a different eval set).

Pre-registration: `PREREGISTRATION_rescore.md`, written before the run. Primary cell `ALL_ID`,
paired `ep4 − ep3`.

In [ ]:
# --- bootstrap -------------------------------------------------------------------
# 🔴 NOTHING that touches HuggingFace or `frame` may be imported here. `HF_HOME` has to be
# set before the first HF import, and its value comes from the PARAMETERS cell below.
import json, logging, os, re, sys, time
from pathlib import Path
import pandas as pd

# 🔴 a papermill kernel does NOT inherit the env's bin/ on PATH (rung 39's scar)
_envbin = str(Path(sys.executable).parent)
if _envbin not in os.environ.get("PATH", "").split(os.pathsep):
    os.environ["PATH"] = _envbin + os.pathsep + os.environ.get("PATH", "")

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s %(message)s",
                    datefmt="%H:%M:%S")
print("python:", sys.executable)

In [ ]:
# --- parameters (RAW LITERALS ONLY — papermill injects BELOW this cell) -------------
SMOKE   = True
EPOCHS  = [3, 4, 5]
N_BOOT  = 4000
KEEP_MERGED = False
STORAGE = "/mnt/storage/uaq_user"

In [ ]:
# --- derived (MUST live BELOW the parameters cell — the rung-16 papermill trap) -----
os.environ["HF_HOME"] = f"{STORAGE}/hf_cache"
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

REPO = f"{STORAGE}/repo_leo"
sys.path.insert(0, f"{REPO}/experiments/45-gen36-data-and-reg/_tools")
sys.path.insert(0, f"{REPO}/experiments/47-epochs-vs-corpus/_tools")
from eval_arm45 import ensure_paths
ensure_paths(REPO)
import eval_arm45 as E45
import eval_arm47 as E
import transformers

cfg = E.Rung47Config(
    repo_root=REPO, work_root=f"{STORAGE}/rung47", data_root=f"{STORAGE}/orena-data",
    frames_cache=f"{STORAGE}/frames_cache", hf_home=f"{STORAGE}/hf_cache",
    epochs=tuple(EPOCHS), n_boot=N_BOOT, keep_merged=KEEP_MERGED, smoke=SMOKE,
)
EXP = Path(REPO) / "experiments" / "47-epochs-vs-corpus"
MANIFEST = Path(REPO) / "experiments" / "splits" / "frame_ood_v1.csv"

# 🔴 A SEPARATE out_dir and run_name. `score_epoch` reuses an existing results.csv, which is
# the right behaviour and exactly the wrong one here: pointing at the 1,283 directory would
# silently return the old answers and call them the 6,252.
BRIDGE_DIR = cfg.run_dir / "eval_full6252"
bridge_run = lambda e: f"{E.RUN}_ep{e}_bridge"

CKPT_ROOT = E.resolve_ckpt_root(cfg)
print("swift    :", (Path(_envbin) / "swift").exists())
print("tfmrs    :", transformers.__version__)
print("arm      :", E.RUN, "· epochs", EPOCHS, "-> steps", [E.EPOCH_STEPS[e] for e in EPOCHS])
print("out_dir  :", BRIDGE_DIR)

In [ ]:
# --- GATE 1: the corpus touched NONE of the 38. RAISES. ----------------------------
# This is the premise the whole notebook rests on. It is measured here, not quoted.
norm = lambda s: re.sub(r"[^A-Za-z0-9]+", "_", s).strip("_")
pat  = re.compile(r"/frames_cache/([A-Za-z0-9_]+)__(\d+)\.jpg")

corpus_vids = set()
for ln in open(f"{STORAGE}/rung47/corpus/train_mntpaths.jsonl", encoding="utf-8"):
    for m in pat.finditer(ln):
        corpus_vids.add(m.group(1))

man = pd.read_csv(MANIFEST, dtype={"dataset": str, "video_id": str})
man["key"] = man.dataset.map(norm) + "__" + man.video_id.map(norm)
by = {s: set(g.key) for s, g in man.groupby("split")}
EVAL_KEYS = by["val_id"] | by["val_ood"]

leak = corpus_vids & EVAL_KEYS
if leak:
    raise AssertionError(f"CONTAMINATED: rung 47's corpus contains {len(leak)} eval videos: "
                         f"{sorted(leak)[:5]} — the 6,252 would report leakage as a result.")
unknown = corpus_vids - by["train"] - EVAL_KEYS
if unknown:
    raise AssertionError(f"{len(unknown)} corpus videos match no manifest row: {sorted(unknown)[:5]}")
if corpus_vids != by["train"]:
    raise AssertionError(f"corpus is {len(corpus_vids)} videos, train is {len(by['train'])}")

print(f"OK legality: corpus = exactly the {len(by['train'])} train videos · "
      f"0 of {len(EVAL_KEYS)} eval videos · 0 unrecognised")

In [ ]:
# --- GATE 2: the eval set, and that every frame it needs is cached. RAISES. --------
from frame.config import BaselineConfig
from frame.data import load_frame_items

items = load_frame_items(BaselineConfig(data_root=Path(cfg.data_root)))
eval_items = [i for i in items if norm(i.dataset) + "__" + norm(i.video_id) in EVAL_KEYS]
BRIDGE_QIDS = {i.request.qID for i in eval_items}  # qID lives on the SDK Request
BRIDGE_VIDEOS = {(i.dataset, i.video_id) for i in eval_items}

if len(BRIDGE_QIDS) != 6252:
    raise AssertionError(f"expected 6,252 questions, got {len(BRIDGE_QIDS)}")
if len(BRIDGE_VIDEOS) != 38:
    raise AssertionError(f"expected 38 videos, got {len(BRIDGE_VIDEOS)}")

split = {"held_videos": BRIDGE_VIDEOS, "held_qids": BRIDGE_QIDS,
         "n_held": len(BRIDGE_QIDS), "n_heico_videos": len(by["val_ood"])}

cov = E45.assert_cache_covers(cfg.to_eval_config(EPOCHS[0]), eval_items)
print(f"OK eval set: {len(BRIDGE_VIDEOS)} videos / {len(BRIDGE_QIDS)} questions · cache {cov}")

In [ ]:
# --- GATE 3: the judge resolves offline, BEFORE anything expensive. RAISES. --------
from transformers import AutoTokenizer
_judge = BaselineConfig().judge_model
try:
    AutoTokenizer.from_pretrained(_judge)
except Exception as exc:
    raise AssertionError(
        f"JUDGE GATE FAILED: {_judge!r} does not resolve offline ({type(exc).__name__}). "
        f"HF_HOME={os.environ.get('HF_HOME')!r}."
    ) from exc
print(f"OK judge gate: {_judge}")

In [ ]:
# --- the SWEEP: merge -> answer the 6,252 -> reclaim the 16 GB ---------------------
# `score_epoch` is bypassed on purpose: it hardwires PRIMARY_EVAL, and the one line that
# matters is `video_filter = None`, which BRIDGE_EVAL selects inside `score()`.
from frame import ledger

gold = ledger.gold_from_frame_parquets(Path(cfg.data_root))
ARM = {}
for e in EPOCHS:
    t0 = time.perf_counter()
    print(f"=== epoch {e} (checkpoint-{E.EPOCH_STEPS[e]}) ===", flush=True)
    ec = cfg.to_eval_config(e)
    ec.eval_set = E45.BRIDGE_EVAL          # -> video_filter = None -> the whole 6,252
    ec.out_dir  = str(BRIDGE_DIR)
    ec.run_name = bridge_run(e)

    existing = BRIDGE_DIR / ec.run_name / "results.csv"
    if existing.exists():
        # 🔴 Reuse is what makes the sweep incremental, and it is also how a 40-question
        # SMOKE table gets scored as if it were the 6,252. Caught in the 2026-08-19 smoke:
        # ep3 came back in 0 s off the previous smoke's file. Count the rows, always.
        n_rows = sum(1 for _ in open(existing, encoding="utf-8")) - 1
        if not SMOKE and n_rows != len(BRIDGE_QIDS):
            raise AssertionError(
                f"{existing} has {n_rows} rows, not {len(BRIDGE_QIDS)}. It is a SMOKE "
                f"artefact — delete {BRIDGE_DIR} before a full run rather than scoring it.")
        print(f"    already answered -> {existing} ({n_rows} rows, no GPU)")
    else:
        ec.merged_dir = str(E.merge_checkpoint(cfg, CKPT_ROOT, e))
        try:
            E45.score(ec, split)
        finally:
            E.reclaim_merged(cfg, e)

    scored = E.score_on_heldout(cfg, E45.arm_results_csv(ec), BRIDGE_QIDS, gold, f"ep{e}")
    ARM[e] = {"epoch": e, "step": E.EPOCH_STEPS[e], "cells": E.cells(scored["strat"]),
              "res": scored["res"], "run_name": ec.run_name}
    c = ARM[e]["cells"]
    print(f"    bucket_mean {c['bucket_mean']:.4f}  acc_ID {c['acc_ID']:.4f}  "
          f"acc_OOD {c['acc_OOD']:.4f}   [{time.perf_counter() - t0:.0f}s]", flush=True)

In [ ]:
# --- 🎯 the pre-registered test: ep4 − ep3 on ALL_ID, 28 clusters -------------------
from frame import metrics as M

def paired(a, b, label):
    ra, rb = ARM[a]["res"], ARM[b]["res"]
    for nm, r in (("a", ra), ("b", rb)):
        for col in ("qID", "video", "correctness"):   # the scored frame calls it `correctness`
            if col not in r.columns:
                raise AssertionError(f"{nm} is missing {col!r}; columns={list(r.columns)}")
    m = ra.merge(rb, on="qID", suffixes=("_a", "_b"))
    # In a full run the two arms must have answered the SAME 6,252. In SMOKE only a
    # handful are answered, so the check becomes "both answered the same ones", which is
    # the property the pairing actually needs.
    if not SMOKE and len(m) != len(BRIDGE_QIDS):
        raise AssertionError(f"paired {len(m)} of {len(BRIDGE_QIDS)} — the arms answered "
                             "different question sets")
    if len(m) != min(len(ra), len(rb)):
        raise AssertionError(f"paired {len(m)} but arms have {len(ra)}/{len(rb)} rows — "
                             "the question sets differ")
    m["video"] = m["video_a"]
    g = m["primary_a"].map(M._leaf_to_group) if "primary_a" in m else m["primary_capability_a"].map(M._leaf_to_group)
    d = m["qID"].map(M._dist_from_qid)
    out = []
    for dd in ("ID", "OOD"):
        for gg in sorted(set(g.dropna())) + ["ALL"]:
            sl = m[(d == dd) & ((g == gg) if gg != "ALL" else True)]
            r = M.paired_delta_ci(sl, correct_a="correctness_a", correct_b="correctness_b",
                                  n_boot=N_BOOT, seed=cfg.seed)
            out.append({"comparison": label, "cell": f"{gg}_{dd}", "n": r["n"],
                        "videos": r["n_videos"], "delta": round(r["delta"], 4),
                        "ci_low": round(r["ci_low"], 4), "ci_high": round(r["ci_high"], 4),
                        "excludes_0": (r["ci_low"] > 0) or (r["ci_high"] < 0),
                        "wins_b": r["wins_b"], "wins_a": r["wins_a"]})
    return out

CI = []
if 3 in ARM and 4 in ARM: CI += paired(3, 4, "ep4 - ep3")
if 4 in ARM and 5 in ARM: CI += paired(4, 5, "ep5 - ep4")
ci_df = pd.DataFrame(CI)
if ci_df.empty:
    # one epoch in the sweep -> nothing to pair. Legitimate in SMOKE, never in a full run.
    if not SMOKE:
        raise AssertionError("no paired comparison produced — a full run needs ep3 AND ep4")
    print("no pair in this sweep (single epoch) — CI skipped")
    ci_df = pd.DataFrame(columns=["comparison", "cell", "n", "videos", "delta",
                                  "ci_low", "ci_high", "excludes_0", "wins_b", "wins_a"])
else:
    print(ci_df.to_string(index=False))

primary = ci_df[(ci_df.comparison == "ep4 - ep3") & (ci_df.cell == "ALL_ID")]
if len(primary):
    p = primary.iloc[0]
    print(f"\n🎯 PRE-REGISTERED CELL — ALL_ID, ep4 − ep3, {p.videos} clusters:")
    print(f"   delta {p.delta:+.4f}  CI [{p.ci_low:+.4f}, {p.ci_high:+.4f}]  "
          f"-> {'EXCLUDES ZERO' if p.excludes_0 else 'INCLUDES ZERO (faithful null)'}")

In [ ]:
# --- write, and say plainly what these numbers are NOT ------------------------------
rows = [{"arm": "47_C_epochs (A2 corpus)", "eval_set": "full 6252 (38 videos)", "epoch": e,
         "ckpt": f"checkpoint-{ARM[e]['step']}", **{k: round(v, 4) for k, v in ARM[e]["cells"].items()}}
        for e in sorted(ARM)]
tbl = pd.DataFrame(rows)
print(tbl.to_string(index=False))

if not SMOKE:
    tbl.to_csv(EXP / "RESULTS_rescore_full6252.csv", index=False)
    ci_df.to_csv(EXP / "RESULTS_rescore_full6252_paired_ci.csv", index=False)
    print("\nwritten:", EXP / "RESULTS_rescore_full6252.csv")

print("""
🔴 NOT COMPARABLE TO:
   · rung 47's own 0.6468  — same checkpoint, 1,283-question eval set
   · rung 42's     0.6744  — different eval set AND it trained on 30 of these 38 videos
   Every citation of a number above names its eval set. ep4 stays the selected epoch;
   it was selected before this run (PREREGISTRATION_rescore.md).
""")